# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.


In [17]:
# import dependencies
import gradio as gr
import openai  # Assuming you need OpenAI here, replace if necessary
from dotenv import load_dotenv
import os

# load environment variables
load_dotenv(override=True)

# fetch the OPENROUTER_API_KEY from environment variables
google_api_key = os.getenv('OPENROUTER_API_KEY')

# Ensure you're using the correct API key to initialize OpenRouter client or any other API client
import openrouter

openrouter.api_key = google_api_key  # Using the OpenRouter API key

# audio output
audio_client = openrouter  # If audio API is part of OpenRouter, update accordingly

def talker(message):
    response = audio_client.audio.speech.create(
        model="gpt-4o-mini-tts",  # Adjust model if necessary
        voice="onyx",  # Change voice if desired
        input=message
    )
    return response.content

# chat function
def chat(message, history, additional_inputs):
    history = [{"role": h['role'], "content": h["content"]} for h in history]
    messages = [{'role': 'system', 'content': system_prompt}] + history + [{'role': 'user', 'content': message}]
    
    # Dynamically choose client based on input
    client = openrouter if additional_inputs == "OpenRouter" else openai  # Use openai if needed
    stream = client.chat.completions.create(
        model="gpt-4o-mini" if additional_inputs == "OpenRouter" else "gemini-2.0-flash",
        messages=messages,
        stream=True
    )
    
    history = history + [{"role": "assistant", "content": ""}]
    response_text = ""
    
    for chunk in stream:
        response_text += chunk.choices[0].delta.content or ''
        history[-1]["content"] = response_text
        yield history, None

    yield history, talker(response_text)  # Pipe to customized Gradio view

# available models
available_models = ["OpenRouter", "Gemini"]

with gr.Blocks() as ui:
    chatbot = gr.Chatbot(height=400)  # Removed the `type` argument
    audioOutput = gr.Audio(autoplay=True)  
    with gr.Row():
        chatInput = gr.Textbox(show_label=False, interactive=True, scale=8, placeholder="Enter your message here...")
        model_dropdown = gr.Dropdown(choices=available_models, value="OpenRouter", interactive=True, scale=2, show_label=False)
    
    chatInput.submit(fn=chat, inputs=[chatInput, chatbot, model_dropdown], outputs=[chatbot, audioOutput])

ui.launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "C:\Users\kwx1494728\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\gradio\queueing.py", line 766, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "C:\Users\kwx1494728\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\gradio\route_utils.py", line 355, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<11 lines>...
    )
    ^
  File "C:\Users\kwx1494728\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\gradio\blocks.py", line 2157, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<8 lines>...
    )
    ^
  File "C:\Users\kwx1494728\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\gradio\blocks.py", line 1646, in call_function
    prediction = await utils.async_iteration(iterat